<a href="https://colab.research.google.com/github/shovo896/Computer_vision/blob/main/CVT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
! pip install qwen_vl_utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 20.9 MB/s eta 0:00:00


In [3]:
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor
from qwen_vl_utils import process_vision_info

In [4]:
model_id = "Qwen/Qwen2-VL-7B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

processor = Qwen2VLProcessor.from_pretrained(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

...

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
def generate_answer(image_path, query, max_new_tokens=256):
    image = Image.open(image_path).convert("RGB")
    sample = {
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": "You are a Vision Language Model. Answer concisely."}]},
            {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": query}]}
        ]
    }

    text_input = processor.apply_chat_template(sample['messages'][1:2], tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(sample['messages'])

    model_inputs = processor(text=[text_input], images=image_inputs, return_tensors="pt").to(device)

    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)

    trimmed_generated_ids = [out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)]

    output_text = processor.batch_decode(trimmed_generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

    return output_text[0]

In [6]:
image_path="/content/Screenshot 2026-02-04 230034.png"
query="Describe this image"
answer=generate_answer(image_path,query)
print("Generate Answer:",answer)

Generate Answer: The image shows a group of people gathered around a wooden table outdoors. On the table, there are several items, including a pink box labeled "Rajveg VIP Sweets," a blue package labeled "Easy Home," and a yellow bag. The people appear to be engaged in an activity, possibly related to the items on the table. The setting seems to be a park or an open area with trees in the background. The group is casually dressed, and the atmosphere appears to be social and relaxed.
